# Data Generation Code

flight = {
    "flight_id": "PA1001",
    "origin": "London",
    "destination": "Hogsmeade",
    "departure_date": "2026-09-26",
    "days_until_departure": 12,
    "base_fare": 135.0,
    "seats_remaining": 18,
    "capacity": 120,
    "route_popularity": 0.91,
    "international": 1
}

structure of data

### Flight Data Generation

The flight dataset is synthetically generated using predefined Canadian and international cities grouped by geographic region.

**Domestic flights** are generated between cities in Western, Central, and Eastern Canada. Every valid origin-destination combination is created in both directions, with two flight instances generated per route. Base fare and aircraft capacity ranges depend on the regional distance of the route (for example, West–West versus West–East).

**International flights** are generated only for predefined Canadian-region-to-international-region combinations, such as Western Canada to the U.S. West Coast, Europe, or East Asia. Each international route is generated in both directions, with two flight instances per direction. Base fare and capacity ranges vary by route group to roughly account for differences in travel distance.

For each route, a base fare and route popularity score are generated and kept consistent across its flight instances. Individual flights receive their own flight ID, departure date, aircraft capacity, and remaining seat count.

Departure dates are randomly selected between **November 1, 2026 and April 1, 2027**, and `days_until_departure` is calculated relative to November 1, 2026.

Each generated flight contains:

* Flight ID
* Origin and destination
* Departure date
* Days until departure
* Base fare
* Seats remaining
* Aircraft capacity
* Route popularity
* Domestic/international indicator

The resulting domestic and international flight records can then be combined and exported as JSON for use by the application.


In [63]:
import pandas as pd
import random

# domestic cities - GPT EDU generated for list of cities
domestic_cities = {
  "west": ["Vancouver", "Calgary", "Edmonton", "Saskatoon", "Regina"],
  "central": ["Winnipeg", "Toronto", "Ottawa", "Montreal", "Quebec City"],
  "east": ["Halifax", "St. John's"]
}

# international cities - GPT EDU generated for list of cities
international_cities = {
  "usa_west": ["Los Angeles", "San Francisco", "Las Vegas"],
  "usa_east": ["New York", "Orlando", "Miami", "Boston", "Washington, D.C."], 
  "mexico": ["Cancun", "Mexico City"],
  "europe": ["London", "Paris", "Amsterdam", "Frankfurt", "Rome", "Barcelona", "Madrid", "Lisbon", "Dublin", "Istanbul"],
  "east_asia": ["Tokyo", "Seoul", "Hong Kong", "Shanghai"],
  "southeast_asia": ["Singapore", "Bangkok"]
}

# date to calculate days till departure from 
reference_date = pd.to_datetime("11-01-2026", format="%m-%d-%Y") 
start_date = reference_date
end_date = pd.to_datetime("04-01-2027", format="%m-%d-%Y") 
departure_dates = pd.date_range(start=start_date, end=end_date)

## Domestic Flight Generation

Domestic flights are generated between cities in Western, Central, and Eastern Canada. Each regional route group has a predefined **base fare range** and **aircraft capacity range**.

### Domestic Flights Pricing & Capacity Logic

1. **West ↔ West, East ↔ East, Central ↔ Central**

   * Base fare: **$150–$350**
   * Capacity: **85–124 passengers**

2. **West ↔ Central**

   * Base fare: **$350–$550**
   * Capacity: **155–190 passengers**

3. **West ↔ East**

   * Base fare: **$500–$750**
   * Capacity: **130–180 passengers**

4. **East ↔ Central**

   * Base fare: **$270–$410**
   * Capacity: **120–175 passengers**

Flights are generated for **both directions** of each route (for example, Vancouver → Toronto and Toronto → Vancouver), with two flight instances per direction. Each route shares the same generated base fare and route popularity, while individual flight details such as departure date, capacity, and seats remaining can vary.

In [74]:
domestic_pricing = {
  "within_province": (150, 300),
  "west_central": (350, 550),
  "west_east": (500, 750),
  "east_central": (270, 410)
}

domestic_capacity = {
  "within_province": (85, 124),
  "west_central": (155, 190),
  "west_east": (130, 180), 
  "east_central": (120, 175)
}

domestic_flights = []
domestic_flight_num = 1000
domestic_flight_id = "PA"+f"""{domestic_flight_num}"""

In [ ]:
# loop over every region - west, central, east (origin)
for origin_region in domestic_cities:
  # for specific origin, also loop through every region (destination)
    for destination_region in domestic_cities:
      # set the group name for each region
      if origin_region == destination_region:
        group = "within_province"
      elif {origin_region, destination_region} == {"west", "central"}:
        group = "west_central"
      elif {origin_region, destination_region} == {"west", "east"}:
        group = "west_east"
      elif {origin_region, destination_region} == {"east", "central"}:
        pricing_group = "east_central"
      # set group from the pricing and capacity dictionaries above
      pricing_range = domestic_pricing[group]
      capacity_range = domestic_capacity[group]
      
      # now loop through each city in origin region
      for origin_city in domestic_cities[origin_region]:
        # for each origin city, loop through all destination cities in destination region
        for destination_city in domestic_cities[destination_region]:
          # only run if origin city and destination city different
          if origin_city != destination_city: 
            # set base fare and route popularity as random values from specified range
            base_fare = random.randint(*pricing_range)
            route_popularity = round(random.uniform(0.5, 0.90), 2)
            # 2 of each route
            for i in range(2):
              capacity = random.randint(*capacity_range)
              departure_date = departure_dates.to_series().sample(1).iloc[0]
              flight = {
                  "flight_id": f"PA{domestic_flight_num}",
                  "origin": origin_city,
                  "destination": destination_city,
                  "departure_date": departure_date.strftime("%m-%d-%Y"),
                  "days_until_departure": (departure_date - reference_date).days,
                  "base_fare": base_fare,
                  "seats_remaining": random.randint(0, capacity),
                  "capacity": capacity,
                  "route_popularity": route_popularity,
                  "international": 0
              }

              domestic_flight_num += 1
              domestic_flights.append(flight)

# International Flight Generation

International flights are generated using predefined combinations of Canadian origin regions and international destination regions. Each route group has a specific **base fare range** and **aircraft capacity range**.

### International — Nearby

1. **West Canada ↔ USA West**

   * Base fare: **$250–$350**
   * Capacity: **160–200 passengers**

2. **Central Canada ↔ USA East**

   * Base fare: **$250–$350**
   * Capacity: **160–200 passengers**

### International — Medium Distance

1. **West Canada ↔ Mexico**

   * Base fare: **$370–$430**
   * Capacity: **120–180 passengers**

2. **Central Canada ↔ Mexico**

   * Base fare: **$500–$800**
   * Capacity: **130–190 passengers**

3. **West Canada ↔ USA East**

   * Base fare: **$400–$600**
   * Capacity: **160–240 passengers**

### International — Long Distance

1. **West Canada ↔ Europe**

   * Base fare: **$900–$1,100**
   * Capacity: **290–320 passengers**

2. **West Canada ↔ East Asia**

   * Base fare: **$1,100–$1,400**
   * Capacity: **300–350 passengers**

3. **West Canada ↔ Southeast Asia**

   * Base fare: **$1,200–$1,500**
   * Capacity: **290–340 passengers**

4. **Central Canada ↔ East Asia**

   * Base fare: **$1,300–$1,500**
   * Capacity: **290–330 passengers**

5. **Central Canada ↔ Southeast Asia**

   * Base fare: **$1,300–$1,700**
   * Capacity: **250–300 passengers**

For each defined Canadian/international city pairing, flights are generated in **both directions**. For example, a West Canada → Europe pairing produces both **Vancouver → London** and **London → Vancouver**. Two flight instances are generated for each direction. Route-level values such as base fare and route popularity remain consistent across the flights for that route, while flight-level attributes such as departure date, capacity, and seats remaining can vary.


In [85]:
international_pricing = {
  ("west", "usa_west"): (250, 350),
  ("central", "usa_east"): (250, 350),
  ("west", "mexico"): (370, 430),
  ("central", "mexico"): (500, 800),
  ("west", "usa_east"): (400, 600),
  ("west", "east_asia"): (1100, 1400),
  ("west", "europe"): (900, 1100),
  ("west", "southeast_asia"): (1200, 1500),
  ("central", "east_asia"): (1300, 1500),
  ("central", "southeast_asia"): (1300, 1700)
}

international_capacity = {
  ("west", "usa_west"): (160, 200),
  ("central", "usa_east"): (160, 200),
  ("west", "mexico"): (120, 180),
  ("central", "mexico"): (130, 190),
  ("west", "usa_east"): (160, 240),
  ("west", "east_asia"): (300, 350),
  ("west", "europe"): (290, 320),
  ("west", "southeast_asia"): (290, 340),
  ("central", "east_asia"): (290, 330),
  ("central", "southeast_asia"): (250, 300)
}

international_flights = []
international_flight_num = 2000
international_flight_id = "PA"+f"""{international_flight_num}"""

In [86]:
# loop through every international route combination - loop this because not every Canadian city will fly internatioanlly (in this case: east canada)
for origin_region, destination_region in international_pricing:
    # get pricing and capacity ranges for this route group
    pricing_range = international_pricing[(origin_region, destination_region)]
    capacity_range = international_capacity[(origin_region, destination_region)]
    
    # loop through every Canadian city in the origin region
    for origin_city in domestic_cities[origin_region]:
        # loop through every international city in the destination region
        for destination_city in international_cities[destination_region]:
            base_fare = random.randint(*pricing_range)
            route_popularity = round(random.uniform(0.5, 0.90), 2)
            # create both directions
            directions = [
                (origin_city, destination_city),
                (destination_city, origin_city)
                ]
            for flight_origin, flight_destination in directions:
                # create 2 flights for each direction
                for i in range(2):
                    capacity = random.randint(*capacity_range)
                    departure_date = departure_dates.to_series().sample(1).iloc[0]
                    flight = {
                        "flight_id": f"PA{international_flight_num}",
                        "origin": flight_origin,
                        "destination": flight_destination,
                        "departure_date": departure_date.strftime("%m-%d-%Y"),
                        "days_until_departure": (departure_date - reference_date).days,
                        "base_fare": base_fare,
                        "seats_remaining": random.randint(0, capacity),
                        "capacity": capacity,
                        "route_popularity": route_popularity,
                        "international": 1
                    }

                    international_flight_num += 1
                    international_flights.append(flight)


### Add both domestic and international flights into one list & save into JSON. 

In [94]:
import json

flights = domestic_flights + international_flights 

with open ("flights.json", "w") as file:
  json.dump(flights, file, indent=2)

In [96]:
len(flights)

1044